# StageBridge
**Cross-modal spatial–snRNA-seq bridge for lung precursor-to-LUAD progression**

---

This notebook is the **entrypoint/orchestrator** for StageBridge data workflows:

1. Build interim AnnData artifacts from GEO raw files (`snrna_*`, `spatial_*`).
2. Run HLCA full-scale mapping for snRNA (`snrna_hlca_latent_full.h5ad` + labels parquet).
3. Run Tangram projection of HLCA-labeled snRNA onto spatial spots.
4. Continue with downstream preprocessing, training, and benchmark/eval cells.

By default, heavy build/mapping steps are disabled behind toggles in section **0A**.
Enable only what you want to run in this session.


## 0 — Imports & Configuration

In [1]:
import json
import shlex
import subprocess
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # change to 'inline' for interactive display
import matplotlib.pyplot as plt
import anndata
import scanpy as sc
from tqdm.auto import tqdm
from IPython.display import Image, display

# ── GPU check ─────────────────────────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
        GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU : {GPU_NAME}  ({GPU_MEM_GB:.1f} GB VRAM)")
        print(f"CUDA: {torch.version.cuda}")
        USE_GPU = True
    else:
        print("PyTorch installed but no CUDA GPU found — running on CPU.")
        USE_GPU = False
except ImportError:
    print("PyTorch not installed.")
    USE_GPU = False

# ── Optional packages ─────────────────────────────────────────────────────
try:
    import squidpy as sq
    SQUIDPY = True
    print(f"squidpy {sq.__version__}")
except ImportError:
    SQUIDPY = False
    print("squidpy not installed — spatial stats steps will be skipped")

try:
    import harmonypy
    HARMONY = True
    print("harmonypy available")
except ImportError:
    HARMONY = False
    print("harmonypy not installed — batch correction step will be skipped")

try:
    import scvi
    SCVI = True
    print(f"scvi-tools {scvi.__version__}")
    if USE_GPU:
        scvi.settings.dl_num_workers = 4   # data loader workers
        # scvi-tools auto-detects GPU; confirm with:
        print(f"  scvi accelerator: {'gpu' if USE_GPU else 'cpu'}")
except ImportError:
    SCVI = False
    print("scvi-tools not installed — scVI/scANVI steps will be skipped")

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sc.settings.verbosity = 1

# ── Repo root on path (if not installed as a package) ─────────────────────
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from stagebridge.logging_utils import configure_root_logger
configure_root_logger()

from stagebridge import config
from stagebridge.preprocessing.harmonize import (
    intersect_genes,
    normalize_log1p,
    select_hvg,
    pca_fit_transform_snrna,
    pca_transform_spatial,
    run_harmony,
    run_umap,
)

FIGURES_DIR = REPO_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nRepo root : {REPO_ROOT}")
print(f"Figures   : {FIGURES_DIR}")


GPU : NVIDIA RTX 4000 Ada Generation  (21.5 GB VRAM)
CUDA: 12.8
squidpy 1.8.1
harmonypy available
scvi-tools 1.4.2
  scvi accelerator: gpu

Repo root : /home/ajbook/projects/StageBridge
Figures   : /home/ajbook/projects/StageBridge/outputs/figures


## 0A — End-to-End Pipeline Runner (AnnData → HLCA → Tangram → Plots)

This section is the operational entrypoint for the full StageBridge data pipeline.
Run the controls cell, then run the runner cell to execute selected stages in order.

In [2]:
# ── Pipeline run controls (notebook entrypoint) ───────────────────────────
PIPELINE_DATA = "local"          # Hydra data config: local | default
PIPELINE_EXPERIMENT = "smoke"    # smoke | full for interim AnnData build scripts
PIPELINE_OVERRIDES = []           # e.g. ["pipeline.max_workers=8"]

# One-switch full pipeline run (recommended for production refreshes).
# When True, it enables all stages below and forces full-mode artifacts.
RUN_PIPELINE_ALL = False

# Stage 1: AnnData build (from GEO-extracted raw files)
RUN_BUILD_SNRNA = False
RUN_BUILD_SPATIAL = False

# Stage 2: HLCA mapping (trained on subset, inferred on full)
RUN_HLCA_MAPPING = False
HLCA_EXPERIMENT = "full"
HLCA_OVERRIDES = []               # e.g. ["hlca.surgery_epochs=200", "hlca.show_progress=true"]

# Stage 3: HLCA quality checks
RUN_HLCA_EVAL = False
HLCA_EVAL_OVERRIDES = []          # e.g. [+hlca_eval.max_query_cells_knn=200000]

# Stage 4: Tangram mapping (HLCA-labeled snRNA -> spatial)
RUN_TANGRAM_MAPPING = False
TANGRAM_EXPERIMENT = "full"
TANGRAM_OVERRIDES = []            # e.g. ["tangram.num_epochs=300", "tangram.device=cuda"]

# Stage 5: Tangram map plotting
RUN_TANGRAM_PLOTS = False
TANGRAM_PLOT_OVERRIDES = []       # e.g. [+tangram_plot.sample_id=GSM..._P1_AAH]

# Stage 6: Build niche tokens from Tangram outputs
RUN_BUILD_NICHE_TOKENS = False
NICHE_TOKEN_CLI_ARGS = []         # e.g. ["--k_neighbors", "8"]

# Stage 7: Build Zarr token bank
RUN_BUILD_NICHE_TOKEN_BANK = False
TOKEN_BANK_CLI_ARGS = []

# Stage 8: QC niche token maps
RUN_QC_NICHE_TOKENS = False
QC_NICHE_SAMPLE_ID = ""
QC_NICHE_CLI_ARGS = []

# Stage 9: Train AIS->MIA with StageBridge + ablations
RUN_TRAIN_POST_TANGRAM = False
TRAIN_POST_TANGRAM_OVERRIDES = [
    "data.max_cells=50000",
    "training.max_epochs=1",
    "training.steps_per_epoch=10",
    "training.val_steps=1",
    "training.batch_cells=128",
    "training.num_ot_pairs=128",
    "training.device=cpu",
    "training.mixed_precision=false",
    "splits.n_folds=2",
    "experiment.baseline_models=[deepsets]",
    "experiment.ablations=[no_context]",
    "training.transition_src=AIS",
    "training.transition_tgt=MIA",
    "training.use_niche_tokens=true",
]

# Stage 10: One-command acceptance chain (tokens -> bank -> qc -> train smoke)
RUN_POST_TANGRAM_ACCEPTANCE = False
POST_TANGRAM_ACCEPTANCE_ARGS = []

# What downstream analysis cells should load after runner execution.
LOAD_EXPERIMENT = PIPELINE_EXPERIMENT   # smoke | full
ATTACH_PRECOMPUTED_HLCA = True

if RUN_PIPELINE_ALL:
    RUN_BUILD_SNRNA = True
    RUN_BUILD_SPATIAL = True
    RUN_HLCA_MAPPING = True
    RUN_HLCA_EVAL = True
    RUN_TANGRAM_MAPPING = True
    RUN_TANGRAM_PLOTS = True
    RUN_BUILD_NICHE_TOKENS = True
    RUN_BUILD_NICHE_TOKEN_BANK = True
    RUN_QC_NICHE_TOKENS = True
    PIPELINE_EXPERIMENT = "full"
    HLCA_EXPERIMENT = "full"
    TANGRAM_EXPERIMENT = "full"
    LOAD_EXPERIMENT = "full"

stage_plan = [
    ("build_snrna_anndata", RUN_BUILD_SNRNA),
    ("build_spatial_anndata", RUN_BUILD_SPATIAL),
    ("map_hlca_full", RUN_HLCA_MAPPING),
    ("eval_hlca_mapping", RUN_HLCA_EVAL),
    ("run_tangram_mapping", RUN_TANGRAM_MAPPING),
    ("plot_tangram_maps", RUN_TANGRAM_PLOTS),
    ("build_niche_tokens", RUN_BUILD_NICHE_TOKENS),
    ("build_niche_token_bank", RUN_BUILD_NICHE_TOKEN_BANK),
    ("qc_niche_tokens", RUN_QC_NICHE_TOKENS),
    ("train_post_tangram", RUN_TRAIN_POST_TANGRAM),
    ("post_tangram_acceptance", RUN_POST_TANGRAM_ACCEPTANCE),
]

print("Pipeline controls configured.")
print(f"  data config      : {PIPELINE_DATA}")
print(f"  build experiment : {PIPELINE_EXPERIMENT}")
print(f"  load experiment  : {LOAD_EXPERIMENT}")
print(f"  HLCA experiment  : {HLCA_EXPERIMENT}")
print(f"  Tangram exp      : {TANGRAM_EXPERIMENT}")
print(f"  run all          : {RUN_PIPELINE_ALL}")
print()
print("Execution plan:")
for stage_name, enabled in stage_plan:
    flag = "ON " if enabled else "off"
    print(f"  [{flag}] {stage_name}")

Controls configured.
  data config         : local
  build experiment    : smoke
  load experiment     : smoke
  run snRNA build     : False
  run spatial build   : False
  run HLCA mapping    : False
  run HLCA eval       : False
  run Tangram mapping : False
  run Tangram plots   : False


In [3]:
NB_RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
PIPELINE_RUNS: dict[str, str] = {}
PIPELINE_SUMMARIES: dict[str, dict | None] = {}
PIPELINE_STAGE_ROWS: list[dict] = []
HLCA_SUMMARY: dict | None = None
TANGRAM_SUMMARY: dict | None = None
TANGRAM_PLOT_SUMMARY: dict | None = None
NICHE_TOKENS_SUMMARY: dict | None = None
TOKEN_BANK_SUMMARY: dict | None = None
QC_NICHE_SUMMARY: dict | None = None
TRAIN_POST_TANGRAM_SUMMARY: dict | None = None
POST_TANGRAM_ACCEPTANCE_SUMMARY: dict | None = None


def _data_root_fallback() -> Path:
    try:
        return config.get_data_root()
    except Exception:
        return Path("/mnt/e/StageBridge_data")


def _run_stagebridge_cmd(cmd: list[str]):
    print("$ " + " ".join(shlex.quote(x) for x in cmd))
    proc = subprocess.Popen(
        cmd,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines: list[str] = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        lines.append(line.rstrip())
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed with exit code {rc}: {' '.join(cmd)}")
    return lines


def _run_stagebridge_cli(script_path: str, overrides: list[str], run_id_prefix: str):
    run_id = f"{run_id_prefix}_{NB_RUN_TS}"
    cmd = [
        sys.executable,
        script_path,
        f"data={PIPELINE_DATA}",
        *overrides,
        f"+run_id={run_id}",
    ]
    lines = _run_stagebridge_cmd(cmd)
    return run_id, lines


def _parse_last_json_line(lines: list[str]) -> dict | None:
    for raw in reversed(lines):
        text = raw.strip()
        if text.startswith("{") and text.endswith("}"):
            try:
                return json.loads(text)
            except json.JSONDecodeError:
                continue
    return None


def _record_stage(stage: str, script: str, run_id: str, summary: dict | None) -> None:
    PIPELINE_RUNS[stage] = run_id
    PIPELINE_SUMMARIES[stage] = summary
    row = {
        "stage": stage,
        "script": script,
        "run_id": run_id,
        "ok": bool(summary.get("ok", True)) if isinstance(summary, dict) else True,
    }
    if isinstance(summary, dict):
        for key in (
            "latent_h5ad",
            "labels_parquet",
            "mapping_h5ad",
            "spatial_h5ad",
            "scores_parquet",
            "celltype_map_png",
            "winner_map_png",
            "out_parquet",
            "out_zarr",
            "ablation_matrix_csv",
        ):
            if key in summary:
                row[key] = summary[key]
    PIPELINE_STAGE_ROWS.append(row)


if RUN_BUILD_SNRNA:
    run_id, lines = _run_stagebridge_cli(
        "scripts/run_snrna_pipeline.py",
        [f"experiment={PIPELINE_EXPERIMENT}", *PIPELINE_OVERRIDES],
        "nb_snrna",
    )
    _record_stage("build_snrna_anndata", "run_snrna_pipeline.py", run_id, _parse_last_json_line(lines))

if RUN_BUILD_SPATIAL:
    run_id, lines = _run_stagebridge_cli(
        "scripts/run_spatial_pipeline.py",
        [f"experiment={PIPELINE_EXPERIMENT}", *PIPELINE_OVERRIDES],
        "nb_spatial",
    )
    _record_stage("build_spatial_anndata", "run_spatial_pipeline.py", run_id, _parse_last_json_line(lines))

if RUN_HLCA_MAPPING:
    run_id, lines = _run_stagebridge_cli(
        "scripts/run_hlca_mapping.py",
        [f"experiment={HLCA_EXPERIMENT}", *HLCA_OVERRIDES],
        "nb_hlca",
    )
    HLCA_SUMMARY = _parse_last_json_line(lines)
    _record_stage("map_hlca_full", "run_hlca_mapping.py", run_id, HLCA_SUMMARY)

if RUN_HLCA_EVAL:
    run_id, lines = _run_stagebridge_cli(
        "scripts/eval_hlca_mapping.py",
        [*HLCA_EVAL_OVERRIDES],
        "nb_hlca_eval",
    )
    _record_stage("eval_hlca_mapping", "eval_hlca_mapping.py", run_id, _parse_last_json_line(lines))

if RUN_TANGRAM_MAPPING:
    run_id, lines = _run_stagebridge_cli(
        "scripts/run_tangram_mapping.py",
        [f"experiment={TANGRAM_EXPERIMENT}", *TANGRAM_OVERRIDES],
        "nb_tangram",
    )
    TANGRAM_SUMMARY = _parse_last_json_line(lines)
    _record_stage("run_tangram_mapping", "run_tangram_mapping.py", run_id, TANGRAM_SUMMARY)

if RUN_TANGRAM_PLOTS:
    run_id, lines = _run_stagebridge_cli(
        "scripts/plot_tangram_maps.py",
        [*TANGRAM_PLOT_OVERRIDES],
        "nb_tangram_plot",
    )
    TANGRAM_PLOT_SUMMARY = _parse_last_json_line(lines)
    _record_stage("plot_tangram_maps", "plot_tangram_maps.py", run_id, TANGRAM_PLOT_SUMMARY)

DATA_ROOT_PIPE = _data_root_fallback()
N_TOK_PARQUET = DATA_ROOT_PIPE / "processed" / "features" / "niche_tokens_full.parquet"
N_BANK_ZARR = DATA_ROOT_PIPE / "processed" / "features" / "niche_token_bank.zarr"
SPATIAL_TG_H5AD = DATA_ROOT_PIPE / "processed" / "tangram" / "spatial_tangram_full.h5ad"
SCORES_PARQUET = DATA_ROOT_PIPE / "processed" / "tangram" / "spatial_tangram_celltype_scores.parquet"

if RUN_BUILD_NICHE_TOKENS:
    run_id = f"nb_niche_{NB_RUN_TS}"
    cmd = [
        sys.executable,
        "scripts/build_niche_tokens.py",
        "--spatial_h5ad", str(SPATIAL_TG_H5AD),
        "--scores_parquet", str(SCORES_PARQUET),
        "--out_parquet", str(N_TOK_PARQUET),
        "--json",
        *NICHE_TOKEN_CLI_ARGS,
    ]
    lines = _run_stagebridge_cmd(cmd)
    NICHE_TOKENS_SUMMARY = _parse_last_json_line(lines)
    _record_stage("build_niche_tokens", "build_niche_tokens.py", run_id, NICHE_TOKENS_SUMMARY)

if RUN_BUILD_NICHE_TOKEN_BANK:
    run_id = f"nb_token_bank_{NB_RUN_TS}"
    cmd = [
        sys.executable,
        "scripts/build_niche_token_bank.py",
        "--niche_tokens_parquet", str(N_TOK_PARQUET),
        "--out_zarr", str(N_BANK_ZARR),
        "--json",
        *TOKEN_BANK_CLI_ARGS,
    ]
    lines = _run_stagebridge_cmd(cmd)
    TOKEN_BANK_SUMMARY = _parse_last_json_line(lines)
    _record_stage("build_niche_token_bank", "build_niche_token_bank.py", run_id, TOKEN_BANK_SUMMARY)

if RUN_QC_NICHE_TOKENS:
    run_id = f"nb_niche_qc_{NB_RUN_TS}"
    cmd = [
        sys.executable,
        "scripts/qc_niche_tokens.py",
        "--niche_tokens_parquet", str(N_TOK_PARQUET),
        "--spatial_h5ad", str(SPATIAL_TG_H5AD),
        "--out_dir", str(REPO_ROOT / "outputs" / "figures" / "niche_tokens"),
        "--json",
        *QC_NICHE_CLI_ARGS,
    ]
    if QC_NICHE_SAMPLE_ID.strip():
        cmd.extend(["--sample_id", QC_NICHE_SAMPLE_ID.strip()])
    lines = _run_stagebridge_cmd(cmd)
    QC_NICHE_SUMMARY = _parse_last_json_line(lines)
    _record_stage("qc_niche_tokens", "qc_niche_tokens.py", run_id, QC_NICHE_SUMMARY)

if RUN_TRAIN_POST_TANGRAM:
    run_id = f"nb_train_post_tangram_{NB_RUN_TS}"
    train_overrides = [
        f"run_name=nb_post_tangram_{NB_RUN_TS}",
        "output_dir=outputs/post_tangram",
        *TRAIN_POST_TANGRAM_OVERRIDES,
        f"training.niche_token_bank_path={N_BANK_ZARR}",
    ]
    run_id, lines = _run_stagebridge_cli(
        "scripts/train_stagebridge.py",
        train_overrides,
        "nb_train_post_tangram",
    )
    TRAIN_POST_TANGRAM_SUMMARY = _parse_last_json_line(lines)
    _record_stage("train_post_tangram", "train_stagebridge.py", run_id, TRAIN_POST_TANGRAM_SUMMARY)

if RUN_POST_TANGRAM_ACCEPTANCE:
    run_id = f"nb_post_tangram_accept_{NB_RUN_TS}"
    cmd = [
        sys.executable,
        "scripts/run_full_step_after_tangram.py",
        "--data_config", PIPELINE_DATA,
        "--json",
        *POST_TANGRAM_ACCEPTANCE_ARGS,
    ]
    lines = _run_stagebridge_cmd(cmd)
    POST_TANGRAM_ACCEPTANCE_SUMMARY = _parse_last_json_line(lines)
    _record_stage(
        "post_tangram_acceptance",
        "run_full_step_after_tangram.py",
        run_id,
        POST_TANGRAM_ACCEPTANCE_SUMMARY,
    )

if not PIPELINE_STAGE_ROWS:
    print("No pipeline scripts executed in this run.")
else:
    stage_df = pd.DataFrame(PIPELINE_STAGE_ROWS)
    print()
    print("Pipeline stage summary:")
    print(stage_df.to_string(index=False))

No pipeline scripts executed in this run.


In [4]:
# ── Resolve data/artifact paths ───────────────────────────────────────────
try:
    DATA_ROOT = config.get_data_root()
    print(f"Data root : {DATA_ROOT}")
except ValueError as e:
    print(f"[ERROR] {e}")
    raise SystemExit(1)


def _artifact_name(prefix: str, experiment: str) -> str:
    exp = str(experiment).lower()
    return f"{prefix}_smoke.h5ad" if exp == "smoke" else f"{prefix}_full.h5ad"


SNRNA_H5AD = DATA_ROOT / "interim" / "anndata" / "snrna" / _artifact_name("snrna", LOAD_EXPERIMENT)
SPATIAL_H5AD = DATA_ROOT / "interim" / "anndata" / "spatial" / _artifact_name("spatial", LOAD_EXPERIMENT)

HLCA_LATENT_H5AD = DATA_ROOT / "processed" / "anndata" / "snrna_hlca_latent_full.h5ad"
HLCA_LABELS_PARQUET = DATA_ROOT / "processed" / "hlca" / "snrna_full_hlca_labels.parquet"
if isinstance(HLCA_SUMMARY, dict):
    HLCA_LATENT_H5AD = Path(HLCA_SUMMARY.get("latent_h5ad", HLCA_LATENT_H5AD))
    HLCA_LABELS_PARQUET = Path(HLCA_SUMMARY.get("labels_parquet", HLCA_LABELS_PARQUET))

TANGRAM_MAP_H5AD = DATA_ROOT / "processed" / "tangram" / "tangram_map_full.h5ad"
TANGRAM_SPATIAL_H5AD = DATA_ROOT / "processed" / "tangram" / "spatial_tangram_full.h5ad"
TANGRAM_SCORES_PARQUET = DATA_ROOT / "processed" / "tangram" / "spatial_tangram_celltype_scores.parquet"
if isinstance(TANGRAM_SUMMARY, dict):
    TANGRAM_MAP_H5AD = Path(TANGRAM_SUMMARY.get("mapping_h5ad", TANGRAM_MAP_H5AD))
    TANGRAM_SPATIAL_H5AD = Path(TANGRAM_SUMMARY.get("spatial_h5ad", TANGRAM_SPATIAL_H5AD))
    TANGRAM_SCORES_PARQUET = Path(TANGRAM_SUMMARY.get("scores_parquet", TANGRAM_SCORES_PARQUET))

NICHE_TOKENS_PARQUET = DATA_ROOT / "processed" / "features" / "niche_tokens_full.parquet"
NICHE_TOKENS_AUDIT = DATA_ROOT / "processed" / "features" / "niche_tokens_full.audit.json"
NICHE_TOKEN_BANK = DATA_ROOT / "processed" / "features" / "niche_token_bank.zarr"
NICHE_TOKEN_BANK_AUDIT = DATA_ROOT / "processed" / "features" / "niche_token_bank.audit.json"

missing = [p for p in (SNRNA_H5AD, SPATIAL_H5AD) if not p.exists()]
if missing:
    lines = ["[ERROR] Required interim h5ad file(s) not found:"]
    for p in missing:
        lines.append(f"  - {p}")
    lines.append("")
    lines.append("Enable build toggles in section 0A and rerun that cell:")
    lines.append("  RUN_BUILD_SNRNA = True")
    lines.append("  RUN_BUILD_SPATIAL = True")
    lines.append("Then execute this path-resolution cell again.")
    raise FileNotFoundError(chr(10).join(lines))

artifact_rows = [
    {"artifact": "snRNA interim h5ad", "path": str(SNRNA_H5AD), "exists": SNRNA_H5AD.exists()},
    {"artifact": "spatial interim h5ad", "path": str(SPATIAL_H5AD), "exists": SPATIAL_H5AD.exists()},
    {"artifact": "HLCA latent h5ad", "path": str(HLCA_LATENT_H5AD), "exists": HLCA_LATENT_H5AD.exists()},
    {"artifact": "HLCA labels parquet", "path": str(HLCA_LABELS_PARQUET), "exists": HLCA_LABELS_PARQUET.exists()},
    {"artifact": "Tangram map h5ad", "path": str(TANGRAM_MAP_H5AD), "exists": TANGRAM_MAP_H5AD.exists()},
    {"artifact": "Tangram spatial h5ad", "path": str(TANGRAM_SPATIAL_H5AD), "exists": TANGRAM_SPATIAL_H5AD.exists()},
    {"artifact": "Tangram score parquet", "path": str(TANGRAM_SCORES_PARQUET), "exists": TANGRAM_SCORES_PARQUET.exists()},
    {"artifact": "Niche tokens parquet", "path": str(NICHE_TOKENS_PARQUET), "exists": NICHE_TOKENS_PARQUET.exists()},
    {"artifact": "Niche tokens audit", "path": str(NICHE_TOKENS_AUDIT), "exists": NICHE_TOKENS_AUDIT.exists()},
    {"artifact": "Niche token bank zarr", "path": str(NICHE_TOKEN_BANK), "exists": NICHE_TOKEN_BANK.exists()},
    {"artifact": "Niche token bank audit", "path": str(NICHE_TOKEN_BANK_AUDIT), "exists": NICHE_TOKEN_BANK_AUDIT.exists()},
]

print()
print("Pipeline artifact status:")
print(pd.DataFrame(artifact_rows).to_string(index=False))

Data root : /mnt/e/StageBridge_data
snRNA h5ad          : /mnt/e/StageBridge_data/interim/anndata/snrna/snrna_smoke.h5ad
Spatial h5ad        : /mnt/e/StageBridge_data/interim/anndata/spatial/spatial_smoke.h5ad
HLCA latent (expect): /mnt/e/StageBridge_data/processed/anndata/snrna_hlca_latent_full.h5ad  [exists=True]
HLCA labels (expect): /mnt/e/StageBridge_data/processed/hlca/snrna_full_hlca_labels.parquet  [exists=True]


## 1 — Load Processed Data

In [5]:
print("Loading snRNA h5ad ...")
adata_rna = anndata.read_h5ad(SNRNA_H5AD)
print(f"  snRNA  shape : {adata_rna.shape}")
print(f"  obs cols     : {list(adata_rna.obs.columns)}")
print(f"  layers       : {list(adata_rna.layers.keys())}")

if "patient_id" not in adata_rna.obs.columns and "donor_id" in adata_rna.obs.columns:
    adata_rna.obs["patient_id"] = adata_rna.obs["donor_id"].astype(str)
if "donor_id" not in adata_rna.obs.columns and "patient_id" in adata_rna.obs.columns:
    adata_rna.obs["donor_id"] = adata_rna.obs["patient_id"].astype(str)

print()
print("Loading spatial h5ad ...")
adata_sp = anndata.read_h5ad(SPATIAL_H5AD)
print(f"  Spatial shape: {adata_sp.shape}")
print(f"  obs cols     : {list(adata_sp.obs.columns)}")
print(f"  obsm keys    : {list(adata_sp.obsm.keys())}")
print(f"  has spatial  : {'spatial' in adata_sp.obsm}")

if "patient_id" not in adata_sp.obs.columns and "donor_id" in adata_sp.obs.columns:
    adata_sp.obs["patient_id"] = adata_sp.obs["donor_id"].astype(str)
if "donor_id" not in adata_sp.obs.columns and "patient_id" in adata_sp.obs.columns:
    adata_sp.obs["donor_id"] = adata_sp.obs["patient_id"].astype(str)


Loading snRNA h5ad ...
  snRNA  shape : (13377, 18082)
  obs cols     : ['barcode', 'donor_id', 'patient_id', 'stage', 'stage_raw', 'gsm_id', 'gsm', 'sample_id', 'modality']
  layers       : ['counts']

Loading spatial h5ad ...
  Spatial shape: (27885, 18120)
  obs cols     : ['spot_id', 'barcode', 'donor_id', 'patient_id', 'stage', 'stage_raw', 'gsm_id', 'gsm', 'sample_id', 'modality']
  obsm keys    : ['spatial']
  has spatial  : True


## 1A — Attach HLCA Outputs (if available)

In [6]:
if ATTACH_PRECOMPUTED_HLCA:
    if HLCA_LABELS_PARQUET.exists():
        labels_df = pd.read_parquet(HLCA_LABELS_PARQUET)
        overlap = adata_rna.obs_names.intersection(labels_df.index)
        print(f"HLCA labels overlap: {len(overlap):,} / {adata_rna.n_obs:,}")

        if len(overlap) > 0:
            cols_to_add = [
                c for c in ["hlca_label", "max_prob", "entropy", "uncertainty"]
                if c in labels_df.columns
            ]
            for col in cols_to_add:
                adata_rna.obs[col] = pd.NA
                adata_rna.obs.loc[overlap, col] = labels_df.loc[overlap, col].values
            if "hlca_label" in cols_to_add:
                adata_rna.obs["hlca_label"] = adata_rna.obs["hlca_label"].astype("category")
                print("Top-10 HLCA labels:")
                print(adata_rna.obs["hlca_label"].value_counts(dropna=True).head(10))
    else:
        print(f"HLCA labels parquet not found: {HLCA_LABELS_PARQUET}")

    if HLCA_LATENT_H5AD.exists():
        adata_hlca_latent = anndata.read_h5ad(HLCA_LATENT_H5AD, backed="r")
        try:
            overlap = adata_rna.obs_names.intersection(adata_hlca_latent.obs_names)
            if len(overlap) == adata_rna.n_obs:
                adata_rna.obsm["X_hlca"] = np.asarray(
                    adata_hlca_latent[adata_rna.obs_names, :].X,
                    dtype=np.float32,
                )
                print(f"Attached adata_rna.obsm['X_hlca']: {adata_rna.obsm['X_hlca'].shape}")
            else:
                print(
                    "Skipping HLCA latent attach due to obs mismatch: "
                    f"{len(overlap):,}/{adata_rna.n_obs:,} overlap"
                )
        finally:
            if getattr(adata_hlca_latent, "isbacked", False) and getattr(adata_hlca_latent, "file", None) is not None:
                adata_hlca_latent.file.close()
    else:
        print(f"HLCA latent h5ad not found: {HLCA_LATENT_H5AD}")
else:
    print("ATTACH_PRECOMPUTED_HLCA=False; skipping HLCA output attachment.")


HLCA labels overlap: 13,377 / 13,377
Top-10 HLCA labels:
hlca_label
T cell lineage        2925
Fibroblast lineage    2637
AT2                   2062
Capillary             1858
Macrophages           1746
Basal                 1039
Ciliated               466
Mast cells             344
Secretory              300
Name: count, dtype: int64
Attached adata_rna.obsm['X_hlca']: (13377, 30)


## 2 — Dataset Summary

In [7]:
def summarise(adata, label):
    print(f"\n{'─'*55}")
    print(f"  {label}")
    print(f"{'─'*55}")
    print(f"  Cells/spots : {adata.n_obs:,}")
    print(f"  Genes       : {adata.n_vars:,}")
    if "sample_id" in adata.obs.columns:
        print(f"  Samples     : {adata.obs['sample_id'].nunique()}")
    if "patient_id" in adata.obs.columns:
        patients = sorted(adata.obs["patient_id"].unique())
        print(f"  Patients    : {len(patients)}  ({', '.join(patients)})")
    if "stage" in adata.obs.columns:
        stage_counts = adata.obs["stage"].value_counts()
        print("  Stages:")
        for stage, n in stage_counts.items():
            print(f"    {stage:<15} {n:>8,}")
    if "spatial" in adata.obsm:
        c = adata.obsm["spatial"]
        print(f"  Spatial     : shape={c.shape}  "
              f"x=[{c[:,0].min():.0f},{c[:,0].max():.0f}]  "
              f"y=[{c[:,1].min():.0f},{c[:,1].max():.0f}]")

summarise(adata_rna, "snRNA-seq (GSE308103)")
summarise(adata_sp,  "Spatial Visium (GSE307534)")


───────────────────────────────────────────────────────
  snRNA-seq (GSE308103)
───────────────────────────────────────────────────────
  Cells/spots : 13,377
  Genes       : 18,082
  Samples     : 4
  Patients    : 2  (P13, P3)
  Stages:
    AIS                5,107
    LUAD               3,877
    MIA                3,510
    Normal               883

───────────────────────────────────────────────────────
  Spatial Visium (GSE307534)
───────────────────────────────────────────────────────
  Cells/spots : 27,885
  Genes       : 18,120
  Samples     : 3
  Patients    : 2  (P2, P8)
  Stages:
    AAH               11,782
    LUAD              10,995
    AIS                5,108
  Spatial     : shape=(27885, 2)  x=[1788,46444]  y=[1008,44097]


## 3 — Gene Intersection

In [8]:
print(f"Before: snRNA {adata_rna.n_vars:,} genes | Spatial {adata_sp.n_vars:,} genes")
adata_rna_int, adata_sp_int = intersect_genes(adata_rna, adata_sp)
print(f"After : {adata_rna_int.n_vars:,} shared genes")

Before: snRNA 18,082 genes | Spatial 18,120 genes
2026-03-02 19:43:52 | INFO     | stagebridge.preprocessing.harmonize | Gene intersection: 18079 common  |  only-A: 3  |  only-B: 41
After : 18,079 shared genes


## 4 — Normalisation (log1p via scanpy)

In [9]:
layer_in = "counts" if "counts" in adata_rna_int.layers else None

print("Normalising snRNA ...")
normalize_log1p(adata_rna_int, layer_in=layer_in, layer_out="log1p")

sp_layer_in = "counts" if "counts" in adata_sp_int.layers else None
print("Normalising spatial ...")
normalize_log1p(adata_sp_int, layer_in=sp_layer_in, layer_out="log1p")

print("Layers — snRNA  :", list(adata_rna_int.layers.keys()))
print("Layers — Spatial:", list(adata_sp_int.layers.keys()))

Normalising snRNA ...
2026-03-02 19:44:00 | INFO     | stagebridge.preprocessing.harmonize | normalize_log1p → adata.layers['log1p']  (target_sum=10000, via scanpy)
Normalising spatial ...
2026-03-02 19:44:00 | INFO     | stagebridge.preprocessing.harmonize | normalize_log1p → adata.layers['log1p']  (target_sum=10000, via scanpy)
Layers — snRNA  : ['counts', 'log1p']
Layers — Spatial: ['counts', 'log1p']


## 5 — Highly Variable Gene Selection (scanpy)

In [10]:
N_HVG = 2000

# seurat_v3 expects raw counts; use counts layer if available
hvg_layer = "counts" if "counts" in adata_rna_int.layers else "log1p"
hvg_flavor = "seurat_v3" if hvg_layer == "counts" else "seurat"

hvg_genes = select_hvg(adata_rna_int, n_hvg=N_HVG, layer=hvg_layer, flavor=hvg_flavor)
print(f"Selected {len(hvg_genes)} HVGs (flavor={hvg_flavor})")
print(f"Example HVGs: {hvg_genes[:10]}")

# Subset both modalities to HVGs present in both
common_hvg = [g for g in hvg_genes if g in set(adata_sp_int.var_names)]
print(f"{len(common_hvg)} of {len(hvg_genes)} HVGs present in spatial data")

adata_rna_hvg = adata_rna_int[:, common_hvg].copy()
adata_sp_hvg  = adata_sp_int[:,  common_hvg].copy()

2026-03-02 19:44:13 | INFO     | stagebridge.preprocessing.harmonize | select_hvg: 2000 HVGs selected from 18079 genes  (flavor=seurat_v3)
Selected 2000 HVGs (flavor=seurat_v3)
Example HVGs: ['A2M', 'AADAC', 'AARD', 'ABCA13', 'ABCA3', 'ABCB5', 'ABCC3', 'ABCC5', 'ABCG1', 'AC007906.2']
2000 of 2000 HVGs present in spatial data


## 6 — PCA

In [11]:
N_PCA = 64

print(f"Fitting PCA ({N_PCA} components) on snRNA "
      f"({adata_rna_hvg.n_obs:,} cells × {adata_rna_hvg.n_vars:,} HVGs) ...")
pca_model = pca_fit_transform_snrna(adata_rna_hvg, n_components=N_PCA, use_layer="log1p")

print(f"Projecting spatial ({adata_sp_hvg.n_obs:,} spots) into snRNA PCA space ...")
pca_transform_spatial(adata_sp_hvg, pca_model, use_layer="log1p")

print(f"snRNA  PCA: {adata_rna_hvg.obsm['X_pca'].shape}")
print(f"Spatial PCA: {adata_sp_hvg.obsm['X_pca'].shape}")

Fitting PCA (64 components) on snRNA (13,377 cells × 2,000 HVGs) ...
2026-03-02 19:44:22 | INFO     | stagebridge.preprocessing.harmonize | Fitting PCA: 64 components on snRNA (shape (13377, 2000), layer='log1p') via scanpy ...
2026-03-02 19:44:22 | INFO     | stagebridge.preprocessing.harmonize | PCA done.  obsm['X_pca'] shape: (13377, 64)  |  cumulative var explained: 39.65%
Projecting spatial (27,885 spots) into snRNA PCA space ...
2026-03-02 19:44:22 | INFO     | stagebridge.preprocessing.harmonize | Spatial PCA projection done.  obsm['X_pca'] shape: (27885, 64)
snRNA  PCA: (13377, 64)
Spatial PCA: (27885, 64)


## 7 — Harmony Batch Correction (across patients)

In [12]:
if HARMONY and "patient_id" in adata_rna_hvg.obs.columns:
    run_harmony(adata_rna_hvg, batch_key="patient_id")
    print(f"Harmony embedding: {adata_rna_hvg.obsm['X_pca_harmony'].shape}")
    HARMONY_DONE = True
else:
    print("Skipping Harmony (harmonypy not installed or no patient_id column).")
    HARMONY_DONE = False

# The basis for downstream steps
EMBED_KEY = "X_pca_harmony" if HARMONY_DONE else "X_pca"

2026-03-02 23:21:12 | INFO     | stagebridge.preprocessing.harmonize | Running Harmony (batch_key='patient_id', n_batches=2) ...


2026-03-02 23:21:12,685 - harmonypy - INFO - Running Harmony (PyTorch on cuda)


2026-03-02 23:21:12 | INFO     | harmonypy | Running Harmony (PyTorch on cuda)


2026-03-02 23:21:12,686 - harmonypy - INFO -   Parameters:


2026-03-02 23:21:12 | INFO     | harmonypy |   Parameters:


...

2026-03-02 23:21:14,501 - harmonypy - INFO - Iteration 10 of 10


2026-03-02 23:21:14 | INFO     | harmonypy | Iteration 10 of 10


2026-03-02 23:21:14,569 - harmonypy - INFO - Stopped before convergence


2026-03-02 23:21:14 | INFO     | harmonypy | Stopped before convergence


ValueError: Value passed for key 'X_pca_harmony' is of incorrect shape. Values of obsm must match dimensions ('obs',) of parent. Value had shape (64,) while it should have had (13377,).

## 8 — UMAP (via scanpy)

In [ ]:
print(f"Computing UMAP for snRNA (basis='{EMBED_KEY}') ...")
run_umap(adata_rna_hvg, basis=EMBED_KEY, n_neighbors=15)
print(f"UMAP done: {adata_rna_hvg.obsm['X_umap'].shape}")

## 9 — Spatial Neighbourhood Graph (squidpy)

In [ ]:
if SQUIDPY and "spatial" in adata_sp_hvg.obsm:
    print("Building spatial neighbourhood graph (squidpy) ...")
    sq.gr.spatial_neighbors(adata_sp_hvg, coord_type="generic", n_neighs=6)
    print("obsp keys:", list(adata_sp_hvg.obsp.keys()))

    # Spatial autocorrelation (Moran's I) on the first PCA component
    print("Computing Moran's I on PC1 ...")
    adata_sp_hvg.obs["PC1"] = adata_sp_hvg.obsm["X_pca"][:, 0]
    sq.gr.spatial_autocorr(
        adata_sp_hvg,
        attr="obs",
        genes=["PC1"],
        mode="moran",
        n_perms=100,
        n_jobs=1,
    )
    moran_df = adata_sp_hvg.uns["moranI"]
    print(moran_df)
else:
    print("Skipping squidpy spatial graph (squidpy not installed or no spatial coords).")

## 10 — Visualisation

In [ ]:
# ── 10a: UMAP coloured by stage ────────────────────────────────────────────
stage_col = "stage" if "stage" in adata_rna_hvg.obs.columns else None

unique_stages = sorted(adata_rna_hvg.obs[stage_col].unique()) if stage_col else []
palette = plt.cm.tab10.colors
stage2color = {s: palette[i % len(palette)] for i, s in enumerate(unique_stages)}

fig, ax = plt.subplots(figsize=(7, 6))
umap = adata_rna_hvg.obsm["X_umap"]
if stage_col:
    stages = adata_rna_hvg.obs[stage_col].values
    for stage in unique_stages:
        m = stages == stage
        ax.scatter(umap[m, 0], umap[m, 1],
                   c=[stage2color[stage]], s=3, alpha=0.4,
                   label=stage, rasterized=True)
    ax.legend(markerscale=4, fontsize=9)
else:
    ax.scatter(umap[:, 0], umap[:, 1], s=3, alpha=0.4, rasterized=True)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
ax.set_title(f"snRNA UMAP — {adata_rna_hvg.n_obs:,} cells  (Harmony+)" if HARMONY_DONE
             else f"snRNA UMAP — {adata_rna_hvg.n_obs:,} cells")
fig.tight_layout()
out = FIGURES_DIR / "umap_snrna_stage.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved: {out}")
plt.close(fig)

In [ ]:
# ── 10b: PCA scatter — spatial in snRNA space ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

def scatter_pca(ax, adata, col, title, alpha=0.3):
    X = adata.obsm["X_pca"]
    if col and col in adata.obs.columns:
        stages = adata.obs[col].values
        for stage in unique_stages:
            m = stages == stage
            if m.any():
                ax.scatter(X[m, 0], X[m, 1], c=[stage2color[stage]],
                           s=4, alpha=alpha, label=stage, rasterized=True)
        ax.legend(markerscale=3, fontsize=8)
    else:
        ax.scatter(X[:, 0], X[:, 1], s=4, alpha=alpha, rasterized=True)
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
    ax.set_title(title, fontsize=11)

scatter_pca(axes[0], adata_rna_hvg, stage_col, "snRNA PCA (PC1 vs PC2)")
scatter_pca(axes[1], adata_sp_hvg,  stage_col, "Spatial projected into snRNA PCA", alpha=0.5)
fig.suptitle("StageBridge — PCA Embedding (log1p, HVGs)", fontsize=13, y=1.01)
fig.tight_layout()
out = FIGURES_DIR / "pca_scatter.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved: {out}")
plt.close(fig)

In [ ]:
# ── 10c: Stage distribution ────────────────────────────────────────────────
if stage_col:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, adata, label in [
        (axes[0], adata_rna_hvg, "snRNA"),
        (axes[1], adata_sp_hvg,  "Spatial"),
    ]:
        if stage_col in adata.obs.columns:
            counts = adata.obs[stage_col].value_counts().sort_index()
            ax.bar(counts.index, counts.values,
                   color=[stage2color.get(s, "gray") for s in counts.index])
            ax.set_title(f"{label} — Stage Distribution")
            ax.set_ylabel("# Cells / Spots")
            for t in ax.get_xticklabels(): t.set_rotation(30)
    fig.tight_layout()
    out = FIGURES_DIR / "stage_distribution.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved: {out}")
    plt.close(fig)

In [ ]:
# ── 10d: Spatial tissue map (squidpy) ─────────────────────────────────────
if SQUIDPY and "spatial" in adata_sp_hvg.obsm:
    # Pick first sample
    first_id = adata_sp_hvg.obs["sample_id"].iloc[0] if "sample_id" in adata_sp_hvg.obs.columns else None
    sub = adata_sp_hvg[adata_sp_hvg.obs["sample_id"] == first_id] if first_id else adata_sp_hvg

    # squidpy spatial scatter
    if stage_col and stage_col in sub.obs.columns:
        color_arg = stage_col
    else:
        sub.obs["PC1"] = sub.obsm["X_pca"][:, 0]
        color_arg = "PC1"

    fig, ax = plt.subplots(figsize=(6, 6))
    sq.pl.spatial_scatter(
        sub,
        color=color_arg,
        ax=ax,
        size=1.2,
        title=f"Spatial tissue — {first_id or 'all'}",
    )
    out = FIGURES_DIR / f"spatial_tissue_{first_id or 'all'}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved: {out}")
    plt.close(fig)
elif "spatial" in adata_sp_hvg.obsm:
    # Manual fallback plot
    coords = adata_sp_hvg.obsm["spatial"]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(coords[:, 0], coords[:, 1], s=6, alpha=0.6, rasterized=True)
    ax.set_title("Spatial tissue map (all samples)")
    ax.set_aspect("equal"); ax.invert_yaxis()
    out = FIGURES_DIR / "spatial_tissue_all.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved: {out}")
    plt.close(fig)
else:
    print("No spatial coordinates available.")

## 10E — Tangram Cell-Type Maps (optional)

In [ ]:
# ── 10e: Show Tangram map figures inline ───────────────────────────────────
# If you want to generate/update these figures, run section 0A with:
#   RUN_TANGRAM_PLOTS = True

celltype_map = None
winner_map = None

if isinstance(TANGRAM_PLOT_SUMMARY, dict):
    celltype_map = Path(TANGRAM_PLOT_SUMMARY.get("celltype_map_png", ""))
    winner_map = Path(TANGRAM_PLOT_SUMMARY.get("winner_map_png", ""))

if not celltype_map or not celltype_map.exists():
    cands = sorted(FIGURES_DIR.glob("tangram_celltype_maps_*.png"), key=lambda p: p.stat().st_mtime, reverse=True)
    celltype_map = cands[0] if cands else None

if not winner_map or not winner_map.exists():
    cands = sorted(FIGURES_DIR.glob("tangram_winner_map_*.png"), key=lambda p: p.stat().st_mtime, reverse=True)
    winner_map = cands[0] if cands else None

if celltype_map and celltype_map.exists():
    print(f"Displaying: {celltype_map}")
    display(Image(filename=str(celltype_map)))
else:
    print("No Tangram per-celltype map found. Set RUN_TANGRAM_PLOTS=True in section 0A and rerun.")

if winner_map and winner_map.exists():
    print(f"Displaying: {winner_map}")
    display(Image(filename=str(winner_map)))
else:
    print("No Tangram winner-label map found. Set RUN_TANGRAM_PLOTS=True in section 0A and rerun.")


## 10F — Niche Tokens & Post-Tangram Training

This section summarizes outputs from the post-Tangram pipeline stages run via section 0A:
1. `build_niche_tokens.py`
2. `build_niche_token_bank.py`
3. `qc_niche_tokens.py`
4. `train_stagebridge.py` (AIS→MIA + ablations)
5. `run_full_step_after_tangram.py` (one-command acceptance)

In [ ]:
# ── 10f: Summarize post-Tangram artifacts and show QC maps ────────────────
post_rows = []
for key in [
    "build_niche_tokens",
    "build_niche_token_bank",
    "qc_niche_tokens",
    "train_post_tangram",
    "post_tangram_acceptance",
]:
    summary = PIPELINE_SUMMARIES.get(key)
    post_rows.append(
        {
            "stage": key,
            "ran": key in PIPELINE_RUNS,
            "ok": bool(summary.get("ok", True)) if isinstance(summary, dict) else None,
            "summary": summary,
        }
    )

print(pd.DataFrame(post_rows)[["stage", "ran", "ok"]].to_string(index=False))

fig_dir = REPO_ROOT / "outputs" / "figures" / "niche_tokens"
for pattern in ["entropy_map_*.png", "tok_AT2_map_*.png", "tok_Ciliated_map_*.png"]:
    cands = sorted(fig_dir.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if cands:
        print(f"Displaying: {cands[0]}")
        display(Image(filename=str(cands[0])))
    else:
        print(f"No figure found for pattern: {pattern}")

## 11 — scVI / scANVI (optional, requires scvi-tools)

In [ ]:
if SCVI:
    import torch

    # ── scVI setup ────────────────────────────────────────────────────────
    # scVI expects raw integer counts in adata.X.
    adata_scvi = adata_rna_hvg.copy()
    if "counts" in adata_scvi.layers:
        adata_scvi.X = adata_scvi.layers["counts"]

    batch_key  = "patient_id" if "patient_id" in adata_scvi.obs.columns else None
    labels_key = "stage"      if "stage"      in adata_scvi.obs.columns else None

    scvi.model.SCVI.setup_anndata(
        adata_scvi,
        layer=None,           # X is already raw counts
        batch_key=batch_key,
        labels_key=labels_key,
    )
    model_scvi = scvi.model.SCVI(
        adata_scvi,
        n_layers=2,
        n_latent=32,
        n_hidden=256,
        gene_likelihood="nb",    # negative binomial — appropriate for UMI counts
        dispersion="gene-batch", # per-gene, per-batch dispersion
    )
    print(model_scvi)

    # ── scANVI setup (semi-supervised with stage labels) ──────────────────
    # scANVI learns a latent space that is batch-corrected AND stage-aware.
    # Central to StageBridge's staging objective.
    if labels_key:
        scvi.model.SCANVI.setup_anndata(
            adata_scvi,
            layer=None,
            batch_key=batch_key,
            labels_key=labels_key,
            unlabeled_category="Unknown",
        )
        model_scanvi = scvi.model.SCANVI(
            adata_scvi,
            n_layers=2,
            n_latent=32,
            n_hidden=256,
            gene_likelihood="nb",
        )
        print(model_scanvi)

    # ── Model output paths (external data root) ───────────────────────────
    SCVI_DIR   = config.scvi_model_dir()
    SCANVI_DIR = config.scanvi_model_dir()

    # ── Training (GPU) ─────────────────────────────────────────────────────
    print()
    if USE_GPU:
        print(f"GPU: {GPU_NAME} ({GPU_MEM_GB:.1f} GB) — training will use CUDA.")
        print()
        print("# ── Run these cells to train: ─────────────────────────────────")
        print(f"model_scvi.train(max_epochs=400, accelerator='gpu', devices=1)")
        print(f"adata_rna_hvg.obsm['X_scVI'] = model_scvi.get_latent_representation()")
        print(f"model_scvi.save('{SCVI_DIR}', overwrite=True)")
        print()
        if labels_key:
            print(f"# Recommended: initialise scANVI from the trained scVI model")
            print(f"model_scanvi = scvi.model.SCANVI.from_scvi_model(model_scvi, labels_key='stage')")
            print(f"model_scanvi.train(max_epochs=20, accelerator='gpu', devices=1)")
            print(f"adata_rna_hvg.obsm['X_scANVI'] = model_scanvi.get_latent_representation()")
            print(f"stage_pred = model_scanvi.predict()   # predicted stage labels")
            print(f"model_scanvi.save('{SCANVI_DIR}', overwrite=True)")
    else:
        print("No GPU detected — scVI training on CPU will be very slow for >10k cells.")
        print("Activate the stagebridge env in a terminal with GPU access and run:")
        print("  jupyter lab StageBridge.ipynb")
else:
    print("scvi-tools not installed — skipping scVI/scANVI setup.")
    print("Install with:  pip install scvi-tools")

## 12 — Summary

In [ ]:
print("=" * 65)
print("  StageBridge preprocessing summary")
print("=" * 65)
print(f"  snRNA  cells × HVGs  : {adata_rna_hvg.shape}")
print(f"  Spatial spots × HVGs : {adata_sp_hvg.shape}")
print(f"  PCA dims             : {adata_rna_hvg.obsm['X_pca'].shape[1]}")
print(f"  Harmony corrected    : {HARMONY_DONE}")
print(f"  UMAP computed        : {'X_umap' in adata_rna_hvg.obsm}")
print(f"  Spatial graph        : {'connectivities' in adata_sp_hvg.obsp}")
print(f"  Figures saved to     : {FIGURES_DIR}")
print("=" * 65)
print()
print("Next steps:")
print("  • Train scVI/scANVI on snRNA (cell 11 above)")
print("  • Run Tangram spatial mapping (stagebridge/models/stagebridge.py)")
print("  • Add cell2location deconvolution")

---

## 13 — Build Latent Representation (HLCA-aligned or PCA fallback)

The `build_latent` helper adds `adata.obsm["X_hlca"]` which is the latent space used for all downstream training.  
If HLCA reference is unavailable it silently falls back to PCA — **training is never blocked**.


In [ ]:
from stagebridge.preprocessing.latent import build_latent, latent_summary
from stagebridge.preprocessing.harmonize import ensure_required_obs_fields

LATENT_KEY = "X_hlca"
N_LATENT_DIM = 64

# Ensure metadata required by downstream training/splitting helpers.
ensure_required_obs_fields(adata_rna_hvg)
if "donor_id" in adata_rna_hvg.obs.columns and "patient_id" in adata_rna_hvg.obs.columns:
    patient_vals = adata_rna_hvg.obs["patient_id"].astype(str)
    donor_vals = adata_rna_hvg.obs["donor_id"].astype(str)
    missing_mask = patient_vals.str.startswith("unknown_patient_id")
    if missing_mask.any():
        adata_rna_hvg.obs.loc[missing_mask, "patient_id"] = donor_vals[missing_mask].values

if LATENT_KEY in adata_rna_hvg.obsm:
    x_lat = np.asarray(adata_rna_hvg.obsm[LATENT_KEY], dtype=np.float32)
    if x_lat.shape[1] > N_LATENT_DIM:
        x_lat = x_lat[:, :N_LATENT_DIM]
        adata_rna_hvg.obsm[LATENT_KEY] = x_lat
    print(f"Using precomputed '{LATENT_KEY}' latent: {x_lat.shape}")
else:
    # Fallback: build HLCA-aligned (or PCA fallback) latent from current object.
    HLCA_REFERENCE = None
    try:
        from stagebridge.io.hlca import load_hlca_reference, hlca_reference_h5ad
        _hlca_path = hlca_reference_h5ad()
        if _hlca_path.exists():
            HLCA_REFERENCE = load_hlca_reference(h5ad_path=_hlca_path)
            print(f"HLCA reference loaded from {_hlca_path}")
        else:
            print(f"HLCA reference not found at {_hlca_path} — using PCA-only alignment.")
    except Exception as _e:
        print(f"HLCA load skipped ({_e}) — using PCA-only alignment.")

    build_latent(
        adata_rna_hvg,
        method="hlca",
        n_components=N_LATENT_DIM,
        output_key=LATENT_KEY,
        pca_layer="log1p",
        hlca_reference=HLCA_REFERENCE,
    )

info = latent_summary(adata_rna_hvg, output_key=LATENT_KEY)
print(f"\nLatent embedding '{LATENT_KEY}':")
for k, v in info.items():
    print(f"  {k:<12}: {v}")


## 14 — Donor-Held-Out Splits

Five-fold (or three-fold for small cohorts) donor-held-out CV splits.  
Each fold guarantees **zero donor overlap** between train / val / test.


In [ ]:
from stagebridge.preprocessing.stage_ontology import normalize_stage_series, CANONICAL_STAGE_ORDER
from stagebridge.training.trainer import (
    build_donor_holdout_splits,
    build_samplers_from_anndata,
    donors_with_min_stage_coverage,
)

STAGE_COL  = "stage"
DONOR_COL  = "patient_id"
N_FOLDS    = 3      # use 5 when ≥10 donors present
SEED       = 42

# Normalise stage labels to canonical ontology
adata_rna_hvg.obs[STAGE_COL] = normalize_stage_series(adata_rna_hvg.obs[STAGE_COL])

obs_stage = np.asarray(adata_rna_hvg.obs[STAGE_COL].astype(str))
obs_donor = np.asarray(adata_rna_hvg.obs[DONOR_COL].astype(str)) if DONOR_COL in adata_rna_hvg.obs.columns \
            else np.array([f"D{i}" for i in range(adata_rna_hvg.n_obs)], dtype=object)

if DONOR_COL not in adata_rna_hvg.obs.columns:
    adata_rna_hvg.obs[DONOR_COL] = obs_donor
    print(f"Warning: '{DONOR_COL}' not found — assigned synthetic donor IDs.")

qualified_donors = donors_with_min_stage_coverage(
    obs_stage=obs_stage,
    obs_donor=obs_donor,
    min_stages=2,
)
print(f"Donors with ≥2 stages: {len(qualified_donors)}  →  {qualified_donors[:10]}")

if len(qualified_donors) < N_FOLDS:
    print(f"[!] Only {len(qualified_donors)} qualified donors — reducing to {max(2, len(qualified_donors))} folds.")
    N_FOLDS = max(2, len(qualified_donors))

splits = build_donor_holdout_splits(donor_ids=qualified_donors, n_folds=N_FOLDS, seed=SEED)

print(f"\nBuilt {N_FOLDS}-fold donor-held-out CV:")
for i, sp in enumerate(splits):
    print(f"  Fold {i}: train={len(sp.train_donors)} donors  "
          f"val={len(sp.val_donors)} donors  "
          f"test={len(sp.test_donors)} donors")

## 15 — Smoke Benchmark Training

Runs **Fold 0** of the StageBridgeModel with a tiny config (2 epochs, CPU) to confirm the full pipeline is wired correctly.  
For a full run use `python scripts/train_stagebridge.py` (with CUDA).

> **Full GPU benchmark**: `python scripts/train_stagebridge.py experiment=full_benchmark`  
> **Smoke CLI run**:       `python scripts/train_stagebridge.py training=smoke model=smoke experiment=smoke`


In [ ]:
import torch
from stagebridge.models.stagebridge import StageBridgeModel
from stagebridge.training.trainer import StageBridgeTrainer
from stagebridge.utils.types import StageBridgeConfig
from stagebridge.utils.seeds import set_global_seed

set_global_seed(SEED)

# ── Resolve device (GPU if available, else CPU) ───────────────────────────
DEVICE = "cuda" if USE_GPU else "cpu"
print(f"Training device: {DEVICE}")

# ── Smoke config — tiny model, 2 epochs, CPU-safe ─────────────────────────
SMOKE_DIM = min(N_LATENT_DIM, 16)   # reduce to 16 for speed

smoke_cfg = StageBridgeConfig(
    input_dim=SMOKE_DIM,
    hidden_dim=32,
    vector_field_hidden_dim=64,
    num_heads=2,
    num_inducing_points=4,
    num_seed_vectors=1,
    num_stages=len(CANONICAL_STAGE_ORDER),
    time_embedding_dim=16,
    stage_embedding_dim=16,
    dropout=0.0,
    ot_epsilon=0.05,
    sinkhorn_iters=5,
    num_ot_pairs=64,
    context_consistency_weight=0.1,
    learning_rate=1e-3,
    weight_decay=1e-4,
    grad_clip_norm=1.0,
    max_epochs=2,
    steps_per_epoch=2,
    val_steps=1,
    patience=2,
    gradient_accumulation_steps=1,
    mixed_precision=False,
    device=DEVICE,
    seed=SEED,
)

# ── Build a lightweight copy of the latent data (SMOKE_DIM components) ────
import anndata as ad

_lat = np.asarray(adata_rna_hvg.obsm[LATENT_KEY], dtype=np.float32)[:, :SMOKE_DIM]
adata_smoke = ad.AnnData(X=np.zeros((_lat.shape[0], 1), dtype=np.float32))
adata_smoke.obsm[LATENT_KEY]  = _lat
adata_smoke.obs[STAGE_COL]    = adata_rna_hvg.obs[STAGE_COL].values
adata_smoke.obs[DONOR_COL]    = adata_rna_hvg.obs[DONOR_COL].values

# ── Build samplers for Fold 0 ──────────────────────────────────────────────
train_sampler, val_sampler, test_sampler = build_samplers_from_anndata(
    adata=adata_smoke,
    split=splits[0],
    latent_key=LATENT_KEY,
    stage_col=STAGE_COL,
    donor_col=DONOR_COL,
    batch_cells=64,
    device=DEVICE,
)
print(f"Train transitions: {train_sampler.available_transitions}")
print(f"Val   transitions: {val_sampler.available_transitions}")
print(f"Test  transitions: {test_sampler.available_transitions}")

# ── Instantiate model and trainer ─────────────────────────────────────────
smoke_model   = StageBridgeModel(config=smoke_cfg)
smoke_trainer = StageBridgeTrainer(model=smoke_model, config=smoke_cfg)

SMOKE_OUTPUT_DIR = FIGURES_DIR.parent / "smoke_run"

smoke_output = smoke_trainer.fit(
    train_sampler=train_sampler,
    val_sampler=val_sampler,
    test_sampler=test_sampler,
    output_dir=SMOKE_OUTPUT_DIR,
    run_name="smoke",
)

print(f"\nSmoke training complete:")
print(f"  Best val loss   : {smoke_output.best_val_loss:.6f}")
print(f"  Epochs run      : {len(smoke_output.history)}")
print(f"  Checkpoint      : {smoke_output.best_checkpoint}")
print(f"  Benchmark keys  : {list(smoke_output.benchmark_metrics.keys())[:6]}")

## 16 — Evaluation Metrics (per transition)

Computes Sinkhorn distance, MMD-RBF, C2ST-AUC, and composition JSD  
for each adjacent stage transition on the held-out test set of Fold 0.


In [ ]:
from stagebridge.training.eval import evaluate_transition
from stagebridge.preprocessing.stage_ontology import stage_to_index

smoke_model.eval()
eval_rows = []

with torch.no_grad():
    for src_stage, tgt_stage in test_sampler.available_transitions:
        x_src, x_tgt = test_sampler.sample_transition_pair(src_stage, tgt_stage, n_cells=128)
        result = evaluate_transition(
            model=smoke_model,
            x_src=x_src,
            x_tgt=x_tgt,
            stage_src=stage_to_index(src_stage),
            stage_tgt=stage_to_index(tgt_stage),
            num_steps=4,
            ot_epsilon=smoke_cfg.ot_epsilon,
            sinkhorn_iters=smoke_cfg.sinkhorn_iters,
        )
        eval_rows.append({
            "transition":       f"{src_stage} → {tgt_stage}",
            "sinkhorn":         result.sinkhorn,
            "mmd_rbf":          result.mmd_rbf,
            "classifier_auc":   result.classifier_auc,
            "jsd_composition":  result.jsd_composition,
            "rank_consistency": result.rank_consistency,
        })

eval_df = pd.DataFrame(eval_rows)
print("Per-transition evaluation (smoke model, Fold 0 test set):\n")
print(eval_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## 17 — Benchmark Summary Figure


In [ ]:
if not eval_df.empty:
    metrics_to_plot = ["sinkhorn", "mmd_rbf", "classifier_auc", "jsd_composition"]
    fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(4 * len(metrics_to_plot), 4))

    for ax, metric in zip(axes, metrics_to_plot):
        bars = ax.bar(
            range(len(eval_df)),
            eval_df[metric],
            color="#1f77b4",
            alpha=0.85,
        )
        ax.set_xticks(range(len(eval_df)))
        ax.set_xticklabels(eval_df["transition"], rotation=25, ha="right", fontsize=8)
        ax.set_title(metric.replace("_", " ").title(), fontsize=10)
        ax.grid(alpha=0.2, axis="y")

    fig.suptitle("StageBridge Smoke Benchmark — Fold 0 Test Metrics", fontsize=12, y=1.02)
    fig.tight_layout()
    out = FIGURES_DIR / "benchmark_smoke_metrics.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved: {out}")
    plt.close(fig)
else:
    print("No eval results — check that test_sampler has available transitions.")

In [ ]:
print("=" * 65)
print("  StageBridge Benchmark — End-to-End Smoke Run Complete")
print("=" * 65)
print()
print(f"  Latent key       : {LATENT_KEY}  ({N_LATENT_DIM}d → smoke {SMOKE_DIM}d)")
print(f"  Donors (CV)      : {len(qualified_donors)}  ({N_FOLDS}-fold donor-held-out)")
print(f"  Model params     : {sum(p.numel() for p in smoke_model.parameters()):,}")
print(f"  Training epochs  : {len(smoke_output.history)}")
print(f"  Best val loss    : {smoke_output.best_val_loss:.6f}")
if smoke_output.benchmark_metrics:
    print(f"  Sinkhorn (mean)  : {smoke_output.benchmark_metrics.get('sinkhorn_mean', float('nan')):.4f}")
    print(f"  MMD-RBF (mean)   : {smoke_output.benchmark_metrics.get('mmd_rbf_mean', float('nan')):.4f}")
    print(f"  C2ST AUC (mean)  : {smoke_output.benchmark_metrics.get('classifier_auc_mean', float('nan')):.4f}")
print()
print("Full GPU run (all models, all folds):")
print("  python scripts/train_stagebridge.py experiment=full_benchmark")
print()
print("Evaluate a saved checkpoint:")
print("  python scripts/eval_stagebridge.py checkpoint=/path/to/checkpoint.pt")
print()
print("Generate poster assets:")
print("  python scripts/make_poster_assets.py outputs/tables/metrics_<run>.json")
print("=" * 65)

---

## 18 — Full Benchmark Training

Runs the complete benchmark: StageBridge + SB-CFM + DeepSets + NoContext + Linear baselines,
3-fold donor-held-out CV, on GPU.

Set `RUN_FULL_BENCHMARK = True` to kick off training (takes ~2–4 hrs on RTX 4000 Ada).
Set `RUN_FULL_BENCHMARK = False` to load results from a previous run.

> **CLI equivalent**:
> ```bash
> python scripts/train_stagebridge.py experiment=full_benchmark data=local run_name=hca_poster
> ```

In [ ]:

# ── Controls ──────────────────────────────────────────────────────────────
RUN_FULL_BENCHMARK = False        # Set True to start full training
FULL_BENCHMARK_RUN_NAME = "hca_poster"
FULL_BENCHMARK_OUTPUT_DIR = "outputs/hca_poster"

# ── Locate existing results (if any) ─────────────────────────────────────
_benchmark_metrics_path = Path(FULL_BENCHMARK_OUTPUT_DIR) / "tables" / f"metrics_{FULL_BENCHMARK_RUN_NAME}.json"
_ablation_csv_path = Path(FULL_BENCHMARK_OUTPUT_DIR) / "tables" / "ablation_matrix.csv"

if RUN_FULL_BENCHMARK:
    print("Launching full benchmark training ...")
    _cmd = [
        sys.executable, "scripts/train_stagebridge.py",
        "experiment=full_benchmark",
        "data=local",
        f"run_name={FULL_BENCHMARK_RUN_NAME}",
        f"output_dir={FULL_BENCHMARK_OUTPUT_DIR}",
    ]
    print("Command:", " ".join(_cmd))
    _proc = subprocess.run(_cmd, capture_output=False, text=True)
    if _proc.returncode != 0:
        print(f"Training exited with code {_proc.returncode}")
    else:
        print("Training complete.")
else:
    if _benchmark_metrics_path.exists():
        print(f"Found existing results: {_benchmark_metrics_path}")
    else:
        print(f"No results found at {_benchmark_metrics_path}")
        print("Set RUN_FULL_BENCHMARK = True to run training, or point FULL_BENCHMARK_OUTPUT_DIR to existing outputs.")


## 19 — Full Benchmark Results

Loads the ablation matrix CSV and displays per-metric tables and bar charts.


In [ ]:

full_results_df = None

if _ablation_csv_path.exists():
    full_results_df = pd.read_csv(_ablation_csv_path)
    # Clean up column names for display
    _display_cols = ["label", "sinkhorn_mean", "mmd_rbf_mean", "classifier_auc_mean",
                     "jsd_composition_mean", "rank_consistency_mean"]
    _show = [c for c in _display_cols if c in full_results_df.columns]
    print(f"Full benchmark results ({len(full_results_df)} variants):")
    display(full_results_df[_show].round(4).style
            .highlight_min(subset=[c for c in _show if "sinkhorn" in c or "mmd" in c or "jsd" in c], color="#c6efce")
            .highlight_max(subset=[c for c in _show if "auc" in c or "rank" in c], color="#c6efce"))
else:
    print("No ablation_matrix.csv found — run Section 18 with RUN_FULL_BENCHMARK = True.")
    full_results_df = pd.DataFrame()


In [ ]:

# ── Bar chart: primary + variants vs baselines ────────────────────────────
if full_results_df is not None and not full_results_df.empty:
    _primary_labels = ["stagebridge", "stagebridge_sb"]
    _baseline_labels = ["deepsets", "no_context", "linear"]
    _plot_labels = [l for l in _primary_labels + _baseline_labels
                    if l in full_results_df["label"].values]
    _plot_df = full_results_df.set_index("label").reindex(_plot_labels)

    _metrics = [("sinkhorn_mean", "Sinkhorn Distance ↓"),
                ("mmd_rbf_mean", "MMD-RBF ↓"),
                ("classifier_auc_mean", "Classifier AUC ↑"),
                ("jsd_composition_mean", "JSD Composition ↓")]
    _metrics = [(m, lab) for m, lab in _metrics if m in _plot_df.columns]

    _colors = ["#1f77b4" if l in _primary_labels else "#aec7e8" for l in _plot_labels]
    fig, axes = plt.subplots(1, len(_metrics), figsize=(5 * len(_metrics), 4))
    if len(_metrics) == 1:
        axes = [axes]

    for ax, (metric, label) in zip(axes, _metrics):
        vals = _plot_df[metric].fillna(0).values
        ax.bar(_plot_labels, vals, color=_colors, alpha=0.85, edgecolor="white", linewidth=0.5)
        ax.set_title(label, fontsize=11)
        ax.set_xticklabels(_plot_labels, rotation=25, ha="right", fontsize=9)
        ax.tick_params(axis="y", labelsize=8)

    fig.suptitle("Full Benchmark: StageBridge vs Baselines", fontsize=13, fontweight="bold")
    fig.tight_layout()
    plt.show()
    print("Blue = StageBridge variants | Light = baselines")


## 20 — Poster Figures

Displays generated figures from the training run: loss curves, UMAP trajectories, Sankey macroflow.
Figures are written to `outputs/hca_poster/figures/` by `train_stagebridge.py`.


In [ ]:

_figures_dir = Path(FULL_BENCHMARK_OUTPUT_DIR) / "figures"

def _show_png(path: Path, title: str = "") -> None:
    if path.exists():
        if title:
            print(f"\n{title}")
        display(Image(filename=str(path), width=900))
    else:
        print(f"[not found] {path.name}")

# ── Loss curves ───────────────────────────────────────────────────────────
for _variant in ["stagebridge", "stagebridge_sb"]:
    _show_png(_figures_dir / f"{FULL_BENCHMARK_RUN_NAME}_{_variant}_loss_curve.png",
              title=f"Loss curve: {_variant}")

# ── UMAP + Sankey for primary model ──────────────────────────────────────
_show_png(_figures_dir / f"{FULL_BENCHMARK_RUN_NAME}_umap_pushforward.png",
          title="UMAP: StageBridge predicted trajectories")
_show_png(_figures_dir / f"{FULL_BENCHMARK_RUN_NAME}_sankey_macroflow.png",
          title="Sankey: macro cell flow")

# ── List all available figures ────────────────────────────────────────────
_pngs = sorted(_figures_dir.glob("*.png")) if _figures_dir.exists() else []
if _pngs:
    print(f"\nAll figures in {_figures_dir} ({len(_pngs)} files):")
    for p in _pngs:
        print(f"  {p.name}")
else:
    print(f"No figures found in {_figures_dir}")
    print("Run Section 18 with RUN_FULL_BENCHMARK = True first.")
